# 📑 CORRECTION : Activité 1 — Audit Post-Mortem du Monolithe FastMarket
## 🎓 Guide de Correction Détaillé & Recommandations d'Architecture Data Cloud

---

### 🎯 Rappel de la Mission :
Derrière chaque plainte métier se cache une **erreur d'architecture technique fondamentale**.  
Voici la grille d'audit corrigée ainsi que les développements techniques attendus pour identifier les causes racines (*Root Causes*) et les goulots d'étranglement (*Bottlenecks*).

---

### 📋 Grille d'Audit Corrigée

| Symptôme Métier | Cause Racine Technique (*Ce qui se passe vraiment sous le capot*) | Composant Saturé (*CPU, RAM, Disque I/O, Locks...*) |
|---|---|---|
| **1. Site gelé à 9h00** *(Rapports BI vs Paniers)* | **Conflit OLTP vs OLAP & Contention de Verrous :** Requêtes analytiques lourdes (`AccessShareLock` prolongés) saturant le pool de connexions et bloquant en cascade les écritures transactionnelles (`RowExclusiveLock` sur `products`/`orders`). | **Locks (Contention de verrous)**, **Pool de connexions**, **CPU** |
| **2. Timeout > 90 jours** *(Requêtes historiques)* | **Absence de partitionnement & Full Table Scan (Modèle Ligne) :** Moteur PostgreSQL forcé de lire 100% des colonnes et des millions de lignes sur disque pour une simple agrégation temporelle. | **Disque I/O (Bande passante disque)**, **RAM (Buffer Cache)** |
| **3. Échec Droit à l'oubli** *(Conformité RGPD)* | **Couplage fort & Contraintes de Clés Étrangères (`ON DELETE RESTRICT`) :** Impossibilité de supprimer un utilisateur sans violer l'intégrité référentielle ou détruire l'historique comptable légal obligatoire. | **Modèle Relationnel / Intégrité Référentielle** |
| **4. Factures sans résultats** *(Scalabilité verticale)* | **Le Mur du Scale-Up :** Doubler la RAM n'augmente ni la concurrence des verrous, ni la latence réseau, ni le parallélisme d'un moteur OLTP monoposte saturé par des locks. | **Architecture Monolithique (Single Point of Contention)**, **IOPS Disque** |

---

### 📚 Références Théoriques Mobilisées
1. **OLTP vs OLAP :** [Datacamp - Différence entre OLTP et OLAP](https://www.datacamp.com/fr/blog/oltp-vs-olap)
2. **Gestion des verrous PostgreSQL :** [Doc Officielle PostgreSQL - Explicit Locking](https://docs.postgresql.fr/15/explicit-locking.html)
3. **Indexation & Full Table Scans :** [Doc PostgreSQL Indexes](https://www.datacamp.com/fr/doc/postgresql/category/postgresql-indexes)
4. **Intégrité référentielle :** [Doc Officielle PostgreSQL - Contraintes FK](https://docs.postgresql.fr/15/ddl-constraints.html#ddl-constraints-fk)
5. **Scalabilité :** [Datacamp - Scaling vertical vs horizontal](https://www.datacamp.com/fr/blog/horizontal-vs-vertical-scaling)

## 📝 Développements & Explications Détaillées (Correction)

---

### 1. Site gelé à 9h00 (Conflits OLTP / OLAP)

* **Cause racine technique :**
  * Les requêtes de reporting marketing exécutent de lourdes jointures (`JOIN order_items`, `products`, `orders`) et agrégations (`GROUP BY category`, `SUM(...)`) qui durent plusieurs minutes.
  * Ces requêtes posent des verrous de lecture (`AccessShareLock`) sur toutes les tables concernées.
  * Un script automatique, une migration, un ORM ou une commande d'analyse lance une opération nécessitant un verrou un peu plus fort (par exemple un CREATE INDEX   CONCURRENTLY qui a besoin d'un ShareUpdateExclusiveLock, ou une réindexation, ou un ALTER TABLE).
    Cette commande ne peut pas s'exécuter car elle est incompatible avec le gros SELECT.

  * **La cascade de blocage :** Si une transaction de maintenance ou une écriture concurrente se place en attente derrière une longue lecture, toutes les requêtes suivantes sont mises en file d'attente. 

  * De plus, les connexions longues monopolisent le pool `max_connections` de PostgreSQL, rejetant les requêtes de l'API web (`500 / 504 Gateway Timeout`).
  * **Impact MVCC :** Les requêtes longues empêchent le processus `VACUUM` de nettoyer les versions de lignes mortes (*dead tuples*), provoquant un gonflement artificiel des tables (*Table Bloat*).

* **Composant saturé :** **Contention de verrous (Table ou Row Locks)**, **Pool de connexions PostgreSQL**, **CPU**.

* **Solution d'architecture recommandée :**
  * Isoler strictement les charges opérationnelles et décisionnelles.
  * Extraire les données de production en continu via le **Change Data Capture (CDC)** avec **Google Cloud Datastream** vers un entrepôt de données dédié : **Google BigQuery**.

---

### 2. Timeout > 90 jours (Volume & Full Table Scans)

* **Cause racine technique :**
  * Dans un SGBD relationnel orienté ligne (*Row-oriented*) comme PostgreSQL, chaque bloc disque contient l'intégralité des colonnes d'un enregistrement (adresse, IP, hash mot de passe, montants, timestamps, etc.).
  * En l'absence d'index composé ou de partitionnement physique par date sur la colonne `created_at`, le moteur est obligé d'exécuter un **Sequential Scan (Full Table Scan)** en lisant des millions de pages disque pour filtrer les lignes.
  * Pour calculer une simple somme sur 90 jours, le moteur doit rapatrier en mémoire vive des gigaoctets de données inutiles (les colonnes textuelles non sollicitées).
  * Le bus I/O disque sature, les buffers débordent et créent des fichiers temporaires sur disque (`temp_files`), menant au dépassement du seuil `statement_timeout` configuré sur l'instance.

* **Composant saturé :** **Disque I/O (Bande passante et IOPS disque)**, **RAM (`shared_buffers` et `work_mem`)**.

* **Solution d'architecture recommandée :**
  * Migrer les données historiques vers un stockage orienté colonnes (*Columnar Storage*) dans **BigQuery** ou des fichiers **Parquet** sur **Google Cloud Storage**.
  * Appliquer un **partitionnement par date** (`PARTITION BY DATE(created_at)`) et un **clustering** (`CLUSTER BY user_id`).

---

### 3. Échec Droit à l'oubli (Couplage fort & Contraintes de clés étrangères)

* **Cause racine technique :**
  * Le schéma monolithique applique une intégrité référentielle rigide : `orders.user_id REFERENCES users(id)` et `order_items.order_id REFERENCES orders(id)`.
  * Lors d'une demande de suppression RGPD (Art. 17 - Droit à l'effacement), la requête `DELETE FROM users WHERE id = X;` est bloquée par l'erreur `violates foreign key constraint` (`ON DELETE RESTRICT` par défaut).
  * Si un `ON DELETE CASCADE` avait été configuré, la suppression de l'utilisateur détruirait en cascade ses commandes et ses paiements associés, ce qui constitue une **infraction légale majeure** (l'article L123-22 du Code de commerce et les obligations fiscales imposent la conservation des factures pendant 10 ans).
  * Le problème vient de la confusion dans le même modèle de données entre l'**Identité de la personne** (données personnelles modifiables/supprimables) et les **Faits transactionnels** (historique financier immuable).

* **Composant saturé :** **Modèle de données relationnel / Contraintes d'intégrité référentielle**.

* **Solution d'architecture recommandée :**
  * Séparer les *Bounded Contexts* (DDD) : isoler le service `Users` du service `Orders`.
  * Dissocier l'identité des commandes via la **pseudonymisation / anonymisation irréversible** :
    * Lors de la suppression d'un utilisateur, ses données PII (`email`, `nom`, `adresse`, `téléphone`) sont purgées de la base `Users` (ou remplacées par un hash / `DELETED_USER_XXXX`).
    * La table des commandes conserve uniquement un identifiant technique anonyme ou utilise le *Crypto-Shredding* (destruction de la clé de chiffrement des données personnelles), préservant la validité des totaux comptables.

---

### 4. Factures sans résultats (Limites de la scalabilité verticale / Scale-up)

* **Cause racine technique :**
  * L'entreprise applique une approche naïve de passage à l'échelle vertical (*Scale-Up* : augmentation des vCPU et de la RAM de la machine virtuelle).
  * **Pourquoi cela ne résout rien :**
    1. **Contention de verrous :** Si 50 transactions attendent un même verrou exclusif sur la table `products`, avoir 64 Go de RAM au lieu de 16 Go ne change rien à la file d'attente séquentielle.
    2. **Latence I/O et verrouillage MVCC :** Le moteur PostgreSQL reste mononœud ; les requêtes concurrentes se disputent le même bus disque et le même contrôleur de stockage.
    3. **Loi des rendements décroissants (Loi d'Amdahl) :** Les opérations sérialisées imposent une limite asymptotique à l'accélération permise par l'ajout de cœurs CPU.
  * Le coût de l'instance augmente de manière exponentielle sans apporter de gain de débit proportionnel.

* **Composant saturé :** **Modèle de traitement centralisé (Single Point of Contention)**, **IOPS Disque**.

* **Solution d'architecture recommandée :**
  * Passer d'une scalabilité verticale (*Scale-Up*) à une **scalabilité horizontale distribuée (*Scale-Out*)** :
    * Découpler le monolithe en microservices avec leur propre base de données (*Database-per-service* sur Cloud SQL).
    * Utiliser un système de messagerie distribuée haut débit (**Google Cloud Pub/Sub**) pour absorber les pics de charge en mode asynchrone.
    * Déporter la puissance de calcul analytique vers un moteur distribué massivement parallèle et élastique (**Google BigQuery**), facturé à l'usage réel.

### 📊 Matrice de Synthèse pour la Direction Technique

| N° | Symptôme Métier Observé | Cause Technique Interne (Moteur PostgreSQL) | Principe d'Architecture Violé | Solution d'Architecture Cible GCP |
|:---:|:---|:---|:---|:---|
| **1** | Crash lors des rapports BI matinaux | Contention de verrous de lecture (`AccessShareLock`) et saturation du pool de connexions | Séparation des charges de travail (*Separation of Concerns : OLTP vs OLAP*) | **Datastream (CDC)** vers **BigQuery** |
| **2** | Timeouts sur requêtes > 90 jours | Balayage séquentiel intégral (*Full Table Scan*) sur stockage en lignes sans partitionnement | Optimisation des accès disques et modélisation analytique | **BigQuery Partitioning & Clustering** (ou BigLake / Parquet sur GCS) |
| **3** | Impossibilité d'appliquer le RGPD | Clés étrangères rigides empêchant la suppression sans détruire les pièces comptables | Découplage de domaine (*Bounded Contexts*) & Immuabilité des faits | Séparation `Users` / `Orders` + **Anonymisation / Crypto-shredding** |
| **4** | Flambée des coûts d'infra sans gain | Saturation de la sérialisation des I/O et des locks face aux limites physiques du Scale-Up | Scalabilité distribuée (*Scale-Out vs Scale-Up*) | Découplage microservices, **Cloud Pub/Sub** et **BigQuery Serverless** |